#### Data Source

Wine Quality Dataset

Source:
UCI Machine Learning Repository

Reference:
Cortez et al. (2009)

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/winequality-red.csv", sep=";")

df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [2]:
display(df.shape)
display(df.columns)
df.info()

(1599, 12)

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality'],
      dtype='object')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


## Initial Dataset Inspection

The red wine dataset contains **1,599 observations** and **12 variables**.

Inspection of the dataset using `df.shape`, `df.columns`, and `df.info()` shows that:

* The dataset contains **1,599 rows** and **12 columns**.
* All columns have **1,599 non-null values**, indicating there are **no missing values**.
* Eleven predictor variables are stored as **`float64`**.
* The target variable, **`quality`**, is stored as **`int64`**, representing the wine quality score assigned by human evaluators.

Since the dataset contains no missing values, no imputation or removal of missing data is required during preprocessing. The next step is to perform exploratory data analysis (EDA) to better understand the distributions of the variables, identify potential outliers, examine relationships between features, and investigate how the physicochemical properties relate to wine quality.

In [3]:
df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000,1599.000000
mean,8.319637,0.527821,0.270976,2.538806,0.087467,15.874922,46.467792,0.996747,3.311113,0.658149,10.422983,5.636023
std,1.741096,0.179060,0.194801,1.409928,0.047065,10.460157,32.895324,0.001887,0.154386,0.169507,1.065668,0.807569
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000
25%,7.100000,0.390000,0.090000,1.900000,0.070000,7.000000,22.000000,0.995600,3.210000,0.550000,9.500000,5.000000
50%,7.900000,0.520000,0.260000,2.200000,0.079000,14.000000,38.000000,0.996750,3.310000,0.620000,10.200000,6.000000
75%,9.200000,0.640000,0.420000,2.600000,0.090000,21.000000,62.000000,0.997835,3.400000,0.730000,11.100000,6.000000
max,15.900000,1.580000,1.000000,15.500000,0.611000,72.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000


The describe() method shows a summary of the numerical attributes.  
Note that the null values are ignored.

In [4]:
df["quality"].value_counts(normalize=True)

quality
5    0.425891
6    0.398999
7    0.124453
4    0.033146
8    0.011257
3    0.006254
Name: proportion, dtype: float64

## Target Variable Distribution

The target variable, **`quality`**, contains six unique wine quality scores ranging from **3 to 8**.

The distribution of quality scores is imbalanced:

* A quality score of **5** represents the largest proportion of the dataset at approximately **42%** of all observations.
* A quality score of **3** represents the smallest proportion at approximately **0.6%** of observations.

The majority of wines in the dataset are rated between **5 and 6**, while extreme quality ratings (3 and 8) are underrepresented.

This imbalance should be considered during model development because machine learning models may become biased toward the more common quality scores. Model evaluation should include appropriate metrics and analysis of predictions across all quality categories to ensure performance is not dominated by the majority class.

In [5]:

duplicates = int(df.duplicated().sum())
print(f"The number of duplicates is {duplicates}.")

The number of duplicates is 240.


## Duplicate Records

Duplicate records were identified using `df.duplicated()`.

The dataset contains **240 duplicate rows**.

These duplicate observations represent approximately **15%** of the dataset. Duplicate records may affect model training by causing certain examples to be weighted more heavily than others, potentially influencing model performance and generalization.

Before model training, duplicate records should be investigated further to determine whether they represent:

* True repeated measurements of the same wine sample
* Data collection artifacts
* Valid observations that should remain in the dataset

A decision on whether to remove duplicates will be made during the data preprocessing stage.

In [7]:
duplicates = df[df.duplicated(keep=False)].sort_values(by=list(df.columns))
duplicates.head(10)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
142,5.2,0.34,0.00,1.8,0.050,27.0,63.0,0.99160,3.68,0.79,14.0,6
144,5.2,0.34,0.00,1.8,0.050,27.0,63.0,0.99160,3.68,0.79,14.0,6
131,5.6,0.50,0.09,2.3,0.049,17.0,99.0,0.99370,3.63,0.63,13.0,5
132,5.6,0.50,0.09,2.3,0.049,17.0,99.0,0.99370,3.63,0.63,13.0,5
1488,5.6,0.54,0.04,1.7,0.049,5.0,13.0,0.99420,3.72,0.58,11.4,5
1491,5.6,0.54,0.04,1.7,0.049,5.0,13.0,0.99420,3.72,0.58,11.4,5
996,5.6,0.66,0.00,2.2,0.087,3.0,11.0,0.99378,3.71,0.63,12.8,7
997,5.6,0.66,0.00,2.2,0.087,3.0,11.0,0.99378,3.71,0.63,12.8,7
829,5.9,0.61,0.08,2.1,0.071,16.0,24.0,0.99376,3.56,0.77,11.1,6
831,5.9,0.61,0.08,2.1,0.071,16.0,24.0,0.99376,3.56,0.77,11.1,6
